In [ ]:
!pip install beautifulsoup4

In [ ]:
PROJECT_ID = "sandbox-373102"
LOCATION = "global"
DATA_STORE_ID = "my-acl-ds-id5"
DISPLAY_NAME = "my-acl-datastore5"
BRANCH_ID = "0"

In [ ]:
import requests
from bs4 import BeautifulSoup

# 1. 설정 및 헤더 (봇 차단 방지용 User-Agent 설정)
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

def get_article_urls(limit=30):
    """목록 페이지에서 상세 게시글 URL을 추출합니다."""
    list_url = "https://dev.to/t/python/latest"
    
    print(f"[{list_url}] 에서 목록을 수집 중입니다...")
    response = requests.get(list_url, headers=headers)
    soup = BeautifulSoup(response.text, 'html.parser')
    
    # 게시글 링크 추출 (dev.to의 구조에 맞춘 selector)
    # class 이름은 사이트 업데이트에 따라 변경될 수 있습니다.
    links = soup.select('.crayons-story__hidden-navigation-link')
    
    article_urls = []
    for link in links:
        full_url = link['href']
        if full_url not in article_urls:
            article_urls.append(full_url)
        if len(article_urls) >= limit:
            break
            
    print(f"총 {len(article_urls)}개의 URL을 확보했습니다.")
    return article_urls

def parse_article_detail(url):
    """상세 페이지에 접속하여 필요한 정보를 파싱합니다."""
    try:
        response = requests.get(url, headers=headers)
        if response.status_code != 200:
            return None
        
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # 1. Title
        title_tag = soup.select_one('h1')
        title = title_tag.get_text(strip=True) if title_tag else "No Title"
        
        # 2. Body (본문 텍스트만 추출)
        body_tag = soup.select_one('#article-body')
        body = body_tag.get_text(strip=True)[:500] + "..." if body_tag else "No Body" # 너무 기니까 500자만
        
        # 3. Author
        author_tag = soup.select_one('.crayons-article__header__meta .crayons-link')
        author = author_tag.get_text(strip=True) if author_tag else "Unknown"
        
        # 4. Date
        date_tag = soup.select_one('time')
        date = date_tag['datetime'] if date_tag else "Unknown"
        
        # 5. Tags & Categories
        # dev.to는 태그 기반이므로, 첫 번째 태그를 category로, 전체를 tags로 간주합니다.
        tag_elements = soup.select('.crayons-tag')
        tags_list = [t.get_text(strip=True).replace('#', '') for t in tag_elements]
        
        categories = tags_list[0] if tags_list else "General"
        tags = ", ".join(tags_list)
        
        return {
            "title": title,
            "body": body,
            "url": url,
            "author": author,
            "categories": categories,
            "tags": tags,
            "date": date
        }
        
    except Exception as e:
        print(f"Error parsing {url}: {e}")
        return None

In [ ]:
import json
from google.cloud import discoveryengine_v1 as discoveryengine
from typing import List
import uuid
def convert_posts_to_documents(posts: List[dict]) -> List[discoveryengine.Document]:
    # Convert WP posts into Discovery Engine Document messages.
    docs: List[discoveryengine.Document] = []
    for post in posts:
        payload = {
            "title": post.get("title"),
            "body": post.get("body"),
            "url": post.get("url"),
            "author": post.get("author"),
            "categories": post.get("categories"),
            "tags": post.get("tags"),
            "date": post.get("date"),
        }
        doc = discoveryengine.Document(
            id=str(uuid.uuid4()),
            json_data=json.dumps(payload),
            #acl_info = discoveryengine.Document.AclInfo(
            #    readers=[{
            #        "principals": [
            #            {"user_id": "abc@example.com"},
            #            {"group_id": "group_1@example.com"}
            #        ]
            #    }]
            #)
        )
        docs.append(doc)
    return docs

In [ ]:
def get_or_create_data_store(
    project_id: str,
    location: str,
    display_name: str,
    data_store_id: str,
) -> discoveryengine.DataStore:
    """Get or create a DataStore."""
    client = discoveryengine.DataStoreServiceClient()
    ds_name = client.data_store_path(project_id, location, data_store_id)
    try:
        result = client.get_data_store(request={"name": ds_name})
        return result
    except:
        parent = client.collection_path(project_id, location, "default_collection")
        operation = client.create_data_store(
            request={
                "parent": parent,
                "data_store": discoveryengine.DataStore(
                    display_name=display_name,
                    #acl_enabled=True,
                    acl_enabled=False,
                    industry_vertical=discoveryengine.IndustryVertical.GENERIC,
                    #identity_mapping_store=identity_mapping_store,
                ),
                "data_store_id": data_store_id,
            }
        )
        result = operation.result()
        return result

In [ ]:
from typing import List
def upload_documents_inline(
    project_id: str,
    location: str,
    data_store_id: str,
    branch_id: str,
    documents: List[discoveryengine.Document],
) -> discoveryengine.ImportDocumentsMetadata:
    """Inline import of Document messages."""
    client = discoveryengine.DocumentServiceClient()
    parent = client.branch_path(
        project=project_id,
        location=location,
        data_store=data_store_id,
        branch=branch_id,
    )
    request = discoveryengine.ImportDocumentsRequest(
        parent=parent,
        inline_source=discoveryengine.ImportDocumentsRequest.InlineSource(
            documents=documents,
        ),
    )
    operation = client.import_documents(request=request)
    operation.result()
    result = operation.metadata
    return result

In [ ]:
import pandas as pd
import time
import random

target_count = 30
urls = get_article_urls(limit=target_count)

posts = []

print("상세 데이터 크롤링 시작...")
for idx, url in enumerate(urls, 1):
    print(f"[{idx}/{len(urls)}] 수집 중: {url}")
        
    article_info = parse_article_detail(url)
    if article_info:
        posts.append(article_info)
        
    # 서버 부하를 줄이기 위해 랜덤하게 0.5~1.5초 대기 (필수 매너)
    time.sleep(random.uniform(0.5, 1.5))

posts

In [ ]:
import google.api_core.exceptions as gcp_exceptions
docs = convert_posts_to_documents(posts)
print(f"Fetched {len(posts)} posts and converted to {len(docs)} documents.")

try:
    data_store = get_or_create_data_store(PROJECT_ID, LOCATION, DISPLAY_NAME, DATA_STORE_ID)
    print("\nEntity Data Store Create Result: ", data_store)

    metadata = upload_documents_inline(
        PROJECT_ID, LOCATION, DATA_STORE_ID, BRANCH_ID, docs
    )
    print(f"Uploaded {metadata.success_count} documents inline.")

except gcp_exceptions.GoogleAPICallError as e:
    print(f"\n--- API Call Failed ---")
    print(f"Server Error Message: {e.message}")
    print(f"Status Code: {e.code}")

except Exception as e:
    print(f"An error occurred: {e}")